DEGRADATION

In [85]:
# 95% CI from Monte Carlo
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:.2e}'.format)

# Data import
file_path = "../Supporting Information 2_PlasticFADE.xlsx"  # CHECK: confirm file path
sheet_name = "Uncertainty"
data_CI = pd.read_excel(file_path, sheet_name=sheet_name, usecols="F:I", skiprows=1)
data_CI = data_CI.iloc[12:18] # Row index minus 3, change this range for other polymers
print(data_CI)

# Parameter estimates
a_i, delta_i, b_i, alpha_i, c_i, beta_i = data_CI.iloc[:, 2].values
# Standard deviations of parameters
a_i_std, delta_i_std, b_i_std, alpha_i_std, c_i_std, beta_i_std = data_CI.iloc[:, 3].values

            Process.1 Parameter.1  Fitted value.1  Standard deviation.1
12  EPS fragmentation         a_i        2.90e-02              4.18e-02
13                NaN     delta_i        9.30e-12              5.45e-01
14                NaN         b_i        1.70e-07              8.83e-07
15                NaN     alpha_i        4.94e+00              2.23e+00
16                NaN         c_i        1.99e+01              1.34e-04
17                NaN      beta_i        1.34e+00              2.69e-01


In [91]:
import pandas as pd
import numpy as np

# File paths
file_path = "../Supporting Information 2_PlasticFADE.xlsx"  # CHECK: confirm file path

def get_parameters(polymer, process):
    """
    Extract fitted values and standard deviations for a specific polymer and process.
    
    Parameters:
    - polymer: e.g., 'HDPE', 'PET', 'LDPE', 'EPS', 'PP', 'PS', 'PVC', 'PLA', 'PA'
    - process: 'fragmentation' or 'degradation'
    
    Returns:
    - params: dictionary of fitted values
    - stds: dictionary of standard deviations
    """
    # Read uncertainty sheet with skiprows=1 to skip the first row
    df_uncertainty = pd.read_excel(file_path, sheet_name="Uncertainty", header=None, skiprows=1)
    
    # Find the row where the process header appears
    process_header_row = None
    for idx, row in df_uncertainty.iterrows():
        cell_value = row.iloc[0]
        if pd.notna(cell_value) and isinstance(cell_value, str):
            if cell_value == f"{polymer} {process}":
                process_header_row = idx
                break
    
    if process_header_row is None:
        raise ValueError(f"Polymer {polymer} with process {process} not found")
    
    if process == 'fragmentation':
        # Parameters on the LEFT side (columns 0-4)
        param_col = 2  # Fitted value column
        std_col = 3    # Standard deviation column
        
        params = {}
        stds = {}
        
        # Look for parameters in rows below the header
        for offset in range(0, 10):  # Check up to 10 rows below header
            row_idx = process_header_row + offset
            if row_idx >= len(df_uncertainty):
                break
                
            row = df_uncertainty.iloc[row_idx]
            param_name = row.iloc[1]  # Parameter name in column 1
            
            if pd.isna(param_name):
                continue
                
            if param_name == 'a_i':
                params['a_i'] = row.iloc[param_col]
                stds['a_i_std'] = row.iloc[std_col]
            elif param_name == 'delta_i':
                params['delta_i'] = row.iloc[param_col]
                stds['delta_i_std'] = row.iloc[std_col]
            elif param_name == 'b_i':
                params['b_i'] = row.iloc[param_col]
                stds['b_i_std'] = row.iloc[std_col]
            elif param_name == 'alpha_i':
                params['alpha_i'] = row.iloc[param_col]
                stds['alpha_i_std'] = row.iloc[std_col]
            elif param_name == 'c_i':
                params['c_i'] = row.iloc[param_col]
                stds['c_i_std'] = row.iloc[std_col]
            elif param_name == 'beta_i':
                params['beta_i'] = row.iloc[param_col]
                stds['beta_i_std'] = row.iloc[std_col]
                break  # Last parameter, can stop searching
        
    elif process == 'degradation':
        # Parameters on the RIGHT side (columns 5-9)
        param_col = 2  # Fitted value column (col 7 = 8th column)
        std_col = 3    # Standard deviation column (col 8 = 9th column)
        
        params = {}
        stds = {}
        
        # Look for parameters in rows below the header
        for offset in range(0, 10):  # Check up to 10 rows below header
            row_idx = process_header_row + offset
            if row_idx >= len(df_uncertainty):
                break
                
            row = df_uncertainty.iloc[row_idx]
            param_name = row.iloc[6]  # Parameter name in column 6 (7th column)
            
            if pd.isna(param_name):
                continue
                
            if param_name == 'x_i':
                params['x_i'] = row.iloc[param_col]
                stds['x_i_std'] = row.iloc[std_col]
            elif param_name == 'tau_i':
                params['tau_i'] = row.iloc[param_col]
                stds['tau_i_std'] = row.iloc[std_col]
            elif param_name == 'y_i':
                params['y_i'] = row.iloc[param_col]
                stds['y_i_std'] = row.iloc[std_col]
            elif param_name == 'theta_i':
                params['theta_i'] = row.iloc[param_col]
                stds['theta_i_std'] = row.iloc[std_col]
            elif param_name == 'z_i':
                params['z_i'] = row.iloc[param_col]
                stds['z_i_std'] = row.iloc[std_col]
            elif param_name == 'eta_i':
                params['eta_i'] = row.iloc[param_col]
                stds['eta_i_std'] = row.iloc[std_col]
                break  # Last parameter, can stop searching
    
    # Print extracted parameters for verification
    print(f"\nExtracted parameters for {polymer} {process}:")
    for key, value in params.items():
        print(f"  {key}: {value:.2e}")
    
    return params, stds

# Choose process
process = 'degradation'  # Change to 'degradation' for degradation runs
polymer = "PE"  

# Read input parameters
data_input = pd.read_excel(file_path, sheet_name="Results", usecols="A:F", skiprows=13)
data_input = data_input[data_input.iloc[:, 0] == polymer]  # Change to your polymer
print(data_input)
data_input.columns = ['Polymer', 'Compartment', 's', 'I_j', 'P_j', 'C_j']

# Get I_j, P_j, C_j from the filtered data
I_j = data_input['I_j'].iloc[0]
P_j = data_input['P_j'].iloc[0]
#C_j = data_input['C_j'].iloc[0] if 'C_j' in data_input.columns else 0
C_j = 2.50E+05
print(f"\nEnvironmental conditions:")
print(f"  I_j (Irradiance): {I_j}")
print(f"  P_j: {P_j}")
print(f"  C_j: {C_j}")

# Monte Carlo setup
N = 10000
np.random.seed(42)
results = []

# Define s values to test (not from results sheet)
s_values = [60000, 6000, 600, 60, 12]



# Get parameters for the chosen polymer and process
params, stds = get_parameters(polymer, process)

# Loop through each s value
for s in s_values:
    print(f"\n--- Processing s = {s} ---")
    
    if process == 'fragmentation':
        a_i = params['a_i']
        delta_i = params['delta_i']
        b_i = params['b_i']
        alpha_i = params['alpha_i']
        c_i = params['c_i']
        beta_i = params['beta_i']
        
        a_i_std = stds['a_i_std']
        delta_i_std = stds['delta_i_std']
        b_i_std = stds['b_i_std']
        alpha_i_std = stds['alpha_i_std']
        c_i_std = stds['c_i_std']
        beta_i_std = stds['beta_i_std']
        
        # Monte Carlo with log-normal distribution
        a_i_samples = np.random.lognormal(np.log(a_i), a_i_std, N)
        delta_i_samples = np.random.lognormal(np.log(delta_i), delta_i_std, N)
        b_i_samples = np.random.lognormal(np.log(b_i), b_i_std, N)
        alpha_i_samples = np.random.lognormal(np.log(alpha_i), alpha_i_std, N)
        c_i_samples = np.random.lognormal(np.log(c_i), c_i_std, N)
        beta_i_samples = np.random.lognormal(np.log(beta_i), beta_i_std, N)
        
        # Compute k_frag for each sample
        k_samples = a_i_samples * (s**delta_i_samples) * (b_i_samples * I_j**alpha_i_samples + c_i_samples * P_j**beta_i_samples)
        k_samples = k_samples[np.isfinite(k_samples)]  # Filter invalid samples
        
        # Point estimate
        k_point = a_i * (s**delta_i) * (b_i * I_j**alpha_i + c_i * P_j**beta_i)
        
    elif process == 'degradation':
        x_i = params['x_i']
        tau_i = params['tau_i']
        y_i = params['y_i']
        theta_i = params['theta_i']
        z_i = params['z_i']
        eta_i = params['eta_i']
        
        x_i_std = stds['x_i_std']
        tau_i_std = stds['tau_i_std']
        y_i_std = stds['y_i_std']
        theta_i_std = stds['theta_i_std']
        z_i_std = stds['z_i_std']
        eta_i_std = stds['eta_i_std']
        
        # Monte Carlo with log-normal distribution
        x_i_samples = np.random.lognormal(np.log(x_i), x_i_std, N)
        tau_i_samples = np.random.lognormal(np.log(tau_i), tau_i_std, N)
        y_i_samples = np.random.lognormal(np.log(y_i), y_i_std, N)
        theta_i_samples = np.random.lognormal(np.log(theta_i), theta_i_std, N)
        z_i_samples = np.random.lognormal(np.log(z_i), z_i_std, N)
        eta_i_samples = np.random.lognormal(np.log(eta_i), eta_i_std, N)
        
        # Compute k_degr for each sample
        k_samples = x_i_samples * (s**tau_i_samples) * (y_i_samples * I_j**theta_i_samples + z_i_samples * C_j**eta_i_samples)
        k_samples = k_samples[np.isfinite(k_samples)]  # Filter invalid samples
        
        # Point estimate
        k_point = x_i * (s**tau_i) * (y_i * I_j**theta_i + z_i * C_j**eta_i)
    
    # 95% CI in log-space
    if np.all(k_samples == 0):
        lower_bound, upper_bound = 0, 0
    else:
        log_k = np.log10(k_samples[k_samples > 0])  # Exclude zeros
        lower_bound = 10 ** np.percentile(log_k, 2.5)
        upper_bound = 10 ** np.percentile(log_k, 97.5)
    
    results.append({
        's': s,
        'k_point': k_point,
        'CI_lower': lower_bound,
        'CI_upper': upper_bound,
        'valid_samples': len(k_samples)
    })
    
    print(f"  k_point = {k_point:.2e}")
    print(f"  95% CI = [{lower_bound:.2e} - {upper_bound:.2e}]")
    print(f"  Invalid samples: {N - len(k_samples)}")

# Display results as DataFrame
results_CI = pd.DataFrame(results)
print("\n" + "="*60)
print("FINAL RESULTS:")
print("="*60)
print(results_CI.to_string())

# Optional: Save to CSV
# results_CI.to_csv(f"{polymer}_{process}_results.csv", index=False)

   Polymer (i) Compartment (j)  SA:V [cm-1]  I_j [W/m2]  P_j [mW]  \
42          PE             Air           25    1.00e+01  4.04e-03   
43          PE         Topsoil           25    1.00e-01  1.15e-03   
44          PE         Subsoil           25    0.00e+00  0.00e+00   
45          PE           Beach           25    1.25e+01  2.64e-02   
46          PE   Water surface           25    1.00e+01  3.30e-03   
47          PE    Water column           25    0.00e+00  1.69e-06   
48          PE        Sediment           25    0.00e+00  0.00e+00   

    C_j [CFU/ml]  
42      5.00e-01  
43      6.70e+08  
44      1.21e+08  
45      1.25e+07  
46      2.50e+05  
47      3.85e+04  
48      4.82e+05  

Environmental conditions:
  I_j (Irradiance): 10.0
  P_j: 0.004042500000000001
  C_j: 250000.0

Extracted parameters for PE degradation:
  x_i: 4.72e-09
  tau_i: 1.35e+00
  y_i: 3.68e-04
  theta_i: 4.99e+00
  z_i: 5.33e-05
  eta_i: 6.51e-01

--- Processing s = 60000 ---
  k_point = 4.69e-01
  

In [117]:
import pandas as pd
import numpy as np

# File paths
file_path = "../Supporting Information 2_PlasticFADE.xlsx"  # CHECK: confirm file path

def get_parameters(polymer, process):
    """
    Extract fitted values and standard deviations for a specific polymer and process.
    """
    # Read uncertainty sheet with skiprows=1 to skip the first row
    df_uncertainty = pd.read_excel(file_path, sheet_name="Uncertainty", header=None, skiprows=1, usecols="A:E")
    
    # Find the row where the process header appears
    process_header_row = None
    for idx, row in df_uncertainty.iterrows():
        cell_value = row.iloc[0]
        if pd.notna(cell_value) and isinstance(cell_value, str):
            if cell_value == f"{polymer} {process}":
                process_header_row = idx
                break
    
    if process_header_row is None:
        raise ValueError(f"Polymer {polymer} with process {process} not found")
    
    if process == 'fragmentation':
        param_col = 2  # Fitted value column
        std_col = 3    # Standard deviation column
        
        params = {}
        stds = {}
        
        for offset in range(0, 10):
            row_idx = process_header_row + offset
            if row_idx >= len(df_uncertainty):
                break
                
            row = df_uncertainty.iloc[row_idx]
            param_name = row.iloc[1]
            
            if pd.isna(param_name):
                continue
                
            if param_name == 'a_i':
                params['a_i'] = row.iloc[param_col]
                stds['a_i_std'] = row.iloc[std_col]
            elif param_name == 'delta_i':
                params['delta_i'] = row.iloc[param_col]
                stds['delta_i_std'] = row.iloc[std_col]
            elif param_name == 'b_i':
                params['b_i'] = row.iloc[param_col]
                stds['b_i_std'] = row.iloc[std_col]
            elif param_name == 'alpha_i':
                params['alpha_i'] = row.iloc[param_col]
                stds['alpha_i_std'] = row.iloc[std_col]
            elif param_name == 'c_i':
                params['c_i'] = row.iloc[param_col]
                stds['c_i_std'] = row.iloc[std_col]
            elif param_name == 'beta_i':
                params['beta_i'] = row.iloc[param_col]
                stds['beta_i_std'] = row.iloc[std_col]
                break
        
    elif process == 'degradation':
        param_col = 2  # Fitted value column
        std_col = 3    # Standard deviation column
        
        params = {}
        stds = {}
        
        for offset in range(0, 10):
            row_idx = process_header_row + offset
            if row_idx >= len(df_uncertainty):
                break
                
            row = df_uncertainty.iloc[row_idx]
            param_name = row.iloc[1]
            
            if pd.isna(param_name):
                continue
                
            if param_name == 'x_i':
                params['x_i'] = row.iloc[param_col]
                stds['x_i_std'] = row.iloc[std_col]
            elif param_name == 'tau_i':
                params['tau_i'] = row.iloc[param_col]
                stds['tau_i_std'] = row.iloc[std_col]
            elif param_name == 'y_i':
                params['y_i'] = row.iloc[param_col]
                stds['y_i_std'] = row.iloc[std_col]
            elif param_name == 'theta_i':
                params['theta_i'] = row.iloc[param_col]
                stds['theta_i_std'] = row.iloc[std_col]
            elif param_name == 'z_i':
                params['z_i'] = row.iloc[param_col]
                stds['z_i_std'] = row.iloc[std_col]
            elif param_name == 'eta_i':
                params['eta_i'] = row.iloc[param_col]
                stds['eta_i_std'] = row.iloc[std_col]
                break
    
    return params, stds

# Read input parameters once
data_input = pd.read_excel(file_path, sheet_name="Results", usecols="A:F", skiprows=13)
data_input.columns = ['Polymer', 'Compartment', 's', 'I_j', 'P_j', 'C_j']

# Environmental conditions (using first row as reference)
#I_j = data_input['I_j'].iloc[0]
#P_j = data_input['P_j'].iloc[0]è
I_j = 0.1
P_j = 1.85E-5
C_j = 6.70E+08

# Monte Carlo setup
N = 10000
np.random.seed(42)

# Define s values to test
s_values = [60000, 6000, 600, 60, 12]

# ============================================================================
# DEGRADATION: Loop over polymers
# ============================================================================
print("="*80)
print("DEGRADATION RESULTS")
print("="*80)

degradation_polymers = ['PP', 'PS', 'PE', 'LDPE', 'HDPE','PET']
all_degradation_results = []

for polymer in degradation_polymers:
    print(f"\n--- Processing {polymer} degradation ---")
    
    # Filter data for this polymer (to get its specific I_j, P_j if needed)
    #data_input_filtered = data_input[data_input.iloc[:, 0] == polymer]
    #if len(data_input_filtered) > 0:
    #    I_j_polymer = data_input_filtered['I_j'].iloc[0]
    #    P_j_polymer = data_input_filtered['P_j'].iloc[0]
    #else:
    I_j_polymer = I_j
    P_j_polymer = P_j
    
    # Get parameters
    try:
        params, stds = get_parameters(polymer, 'degradation')
    except ValueError as e:
        print(f"  Skipping {polymer}: {e}")
        continue
    
    x_i = params['x_i']
    tau_i = params['tau_i']
    y_i = params['y_i']
    theta_i = params['theta_i']
    z_i = params['z_i']
    eta_i = params['eta_i']
    
    x_i_std = stds['x_i_std']
    tau_i_std = stds['tau_i_std']
    y_i_std = stds['y_i_std']
    theta_i_std = stds['theta_i_std']
    z_i_std = stds['z_i_std']
    eta_i_std = stds['eta_i_std']
    
    for s in s_values:
        # Monte Carlo with log-normal distribution
        x_i_samples = np.random.lognormal(np.log(x_i), x_i_std, N)
        tau_i_samples = np.random.lognormal(np.log(tau_i), tau_i_std, N)
        y_i_samples = np.random.lognormal(np.log(y_i), y_i_std, N)
        theta_i_samples = np.random.lognormal(np.log(theta_i), theta_i_std, N)
        z_i_samples = np.random.lognormal(np.log(z_i), z_i_std, N)
        eta_i_samples = np.random.lognormal(np.log(eta_i), eta_i_std, N)
        
        # Compute k_degr for each sample
        k_samples = x_i_samples * (s**tau_i_samples) * (y_i_samples * I_j_polymer**theta_i_samples + z_i_samples * C_j**eta_i_samples)
        k_samples = k_samples[np.isfinite(k_samples)]
        
        # Point estimate
        k_point = x_i * (s**tau_i) * (y_i * I_j_polymer**theta_i + z_i * C_j**eta_i)
        
        # 95% CI in log-space
        if np.all(k_samples == 0):
            lower_bound, upper_bound = 0, 0
        else:
            log_k = np.log10(k_samples[k_samples > 0])
            lower_bound = 10 ** np.percentile(log_k, 2.5)
            upper_bound = 10 ** np.percentile(log_k, 97.5)
        
        all_degradation_results.append({
            'Polymer': polymer,
            'Process': 'Degradation',
            's': s,
            'k_point': k_point,
            'CI_lower': lower_bound,
            'CI_upper': upper_bound
        })
    
    print(f"  Completed {polymer}")

# ============================================================================
# FRAGMENTATION: Loop over polymers
# ============================================================================
print("\n" + "="*80)
print("FRAGMENTATION RESULTS")
print("="*80)

fragmentation_polymers = ['PP', 'PS', 'EPS']
all_fragmentation_results = []

for polymer in fragmentation_polymers:
    print(f"\n--- Processing {polymer} fragmentation ---")
    
    # Filter data for this polymer
    #data_input_filtered = data_input[data_input.iloc[:, 0] == polymer]
    #if len(data_input_filtered) > 0:
    #    I_j_polymer = data_input_filtered['I_j'].iloc[0]
    #    P_j_polymer = data_input_filtered['P_j'].iloc[0]
    #else:
    I_j_polymer = I_j
    P_j_polymer = P_j
    
    # Get parameters
    try:
        params, stds = get_parameters(polymer, 'fragmentation')
    except ValueError as e:
        print(f"  Skipping {polymer}: {e}")
        continue
    
    a_i = params['a_i']
    delta_i = params['delta_i']
    b_i = params['b_i']
    alpha_i = params['alpha_i']
    c_i = params['c_i']
    beta_i = params['beta_i']
    
    a_i_std = stds['a_i_std']
    delta_i_std = stds['delta_i_std']
    b_i_std = stds['b_i_std']
    alpha_i_std = stds['alpha_i_std']
    c_i_std = stds['c_i_std']
    beta_i_std = stds['beta_i_std']
    
    for s in s_values:
        # Monte Carlo with log-normal distribution
        a_i_samples = np.random.lognormal(np.log(a_i), a_i_std, N)
        delta_i_samples = np.random.lognormal(np.log(delta_i), delta_i_std, N)
        b_i_samples = np.random.lognormal(np.log(b_i), b_i_std, N)
        alpha_i_samples = np.random.lognormal(np.log(alpha_i), alpha_i_std, N)
        c_i_samples = np.random.lognormal(np.log(c_i), c_i_std, N)
        beta_i_samples = np.random.lognormal(np.log(beta_i), beta_i_std, N)
        
        # Compute k_frag for each sample
        k_samples = a_i_samples * (s**delta_i_samples) * (b_i_samples * I_j_polymer**alpha_i_samples + c_i_samples * P_j_polymer**beta_i_samples)
        k_samples = k_samples[np.isfinite(k_samples)]
        
        # Point estimate
        k_point = a_i * (s**delta_i) * (b_i * I_j_polymer**alpha_i + c_i * P_j_polymer**beta_i)
        
        # 95% CI in log-space
        if np.all(k_samples == 0):
            lower_bound, upper_bound = 0, 0
        else:
            log_k = np.log10(k_samples[k_samples > 0])
            lower_bound = 10 ** np.percentile(log_k, 2.5)
            upper_bound = 10 ** np.percentile(log_k, 97.5)
        
        all_fragmentation_results.append({
            'Polymer': polymer,
            'Process': 'Fragmentation',
            's': s,
            'k_point': k_point,
            'CI_lower': lower_bound,
            'CI_upper': upper_bound
        })
    
    print(f"  Completed {polymer}")

# ============================================================================
# FINAL TABLES
# ============================================================================

# Combine all results
all_results = pd.DataFrame(all_degradation_results + all_fragmentation_results)

# Pivot tables for better visualization
print("\n" + "="*80)
print("DEGRADATION - Point estimates (k_point)")
print("="*80)
degradation_pivot = pd.DataFrame(all_degradation_results).pivot(index='Polymer', columns='s', values='k_point')
print(degradation_pivot.to_string())

print("\n" + "="*80)
print("DEGRADATION - 95% CI Lower Bounds")
print("="*80)
degradation_lower = pd.DataFrame(all_degradation_results).pivot(index='Polymer', columns='s', values='CI_lower')
print(degradation_lower.to_string())

print("\n" + "="*80)
print("DEGRADATION - 95% CI Upper Bounds")
print("="*80)
degradation_upper = pd.DataFrame(all_degradation_results).pivot(index='Polymer', columns='s', values='CI_upper')
print(degradation_upper.to_string())

print("\n" + "="*80)
print("FRAGMENTATION - Point estimates (k_point)")
print("="*80)
fragmentation_pivot = pd.DataFrame(all_fragmentation_results).pivot(index='Polymer', columns='s', values='k_point')
print(fragmentation_pivot.to_string())

print("\n" + "="*80)
print("FRAGMENTATION - 95% CI Lower Bounds")
print("="*80)
fragmentation_lower = pd.DataFrame(all_fragmentation_results).pivot(index='Polymer', columns='s', values='CI_lower')
print(fragmentation_lower.to_string())

print("\n" + "="*80)
print("FRAGMENTATION - 95% CI Upper Bounds")
print("="*80)
fragmentation_upper = pd.DataFrame(all_fragmentation_results).pivot(index='Polymer', columns='s', values='CI_upper')
print(fragmentation_upper.to_string())

# ============================================================================
# MIN LOWER BOUNDS AND MAX UPPER BOUNDS
# ============================================================================

print("\n" + "="*80)
print("MIN LOWER BOUNDS AND MAX UPPER BOUNDS")
print("="*80)

if all_degradation_results:
    print(f"Degradation - Min CI_lower: {min([r['CI_lower'] for r in all_degradation_results if r['CI_lower'] > 0]):.2e}")
    print(f"Degradation - Max CI_upper: {max([r['CI_upper'] for r in all_degradation_results if r['CI_upper'] > 0]):.2e}")

if all_fragmentation_results:
    print(f"Fragmentation - Min CI_lower: {min([r['CI_lower'] for r in all_fragmentation_results if r['CI_lower'] > 0]):.2e}")
    print(f"Fragmentation - Max CI_upper: {max([r['CI_upper'] for r in all_fragmentation_results if r['CI_upper'] > 0]):.2e}")

# Optional: Save to CSV
# all_results.to_csv("all_monte_carlo_results.csv", index=False)

DEGRADATION RESULTS

--- Processing PP degradation ---
  Completed PP

--- Processing PS degradation ---
  Completed PS

--- Processing PE degradation ---
  Completed PE

--- Processing LDPE degradation ---
  Completed LDPE

--- Processing HDPE degradation ---
  Completed HDPE

--- Processing PET degradation ---
  Completed PET

FRAGMENTATION RESULTS

--- Processing PP fragmentation ---
  Completed PP

--- Processing PS fragmentation ---
  Completed PS

--- Processing EPS fragmentation ---
  Completed EPS

DEGRADATION - Point estimates (k_point)
s          12       60       600      6000     60000
Polymer                                             
HDPE    3.27e-04 3.27e-04 3.27e-04 3.27e-04 3.27e-04
LDPE    9.52e-04 9.52e-04 9.52e-04 9.52e-04 9.52e-04
PE      3.98e-06 3.48e-05 7.75e-04 1.73e-02 3.84e-01
PET     1.30e-04 4.65e-04 2.87e-03 1.78e-02 1.10e-01
PP      7.53e-04 7.70e-04 7.95e-04 8.20e-04 8.46e-04
PS      7.45e-04 1.10e-03 1.93e-03 3.38e-03 5.92e-03

DEGRADATION - 95% CI Lo

/var/folders/dk/y_33q_hj5yqgz652xn7q13cm0000gn/T/ipykernel_16469/3266885590.py:253: RuntimeWarning: overflow encountered in power
  k_samples = a_i_samples * (s**delta_i_samples) * (b_i_samples * I_j_polymer**alpha_i_samples + c_i_samples * P_j_polymer**beta_i_samples)
